# Notebook 4: Data Balancing for Exercise Classification
## Handling Class Imbalance with SMOTE and ADASYN

---

## 📋 Overview

**Objectives:**
1. Understand the class imbalance problem
2. Analyze our dataset's imbalance
3. Apply SMOTE (Synthetic Minority Oversampling Technique)
4. Apply ADASYN (Adaptive Synthetic Sampling)
5. Compare balancing strategies
6. Prepare balanced data for training

---

## 🎯 Why Balance Data?

### The Class Imbalance Problem

**Scenario:** Imagine your dataset has:
- 1000 correct form examples
- 100 incorrect form examples

**What Happens?**
```
Model learns: "Just predict 'correct' every time!"
Result: 90.9% accuracy... but USELESS!
```

**The Accuracy Paradox:**
- High accuracy, but model ignores minority class
- Defeats the purpose: We NEED to detect incorrect form!
- Minority class is often the most important class

### Why Not Just Collect More Data?

**Reality Check:**
- Incorrect form examples are harder to collect
- Real-world data is naturally imbalanced
- Data collection is expensive and time-consuming

**Solution:** Synthetic data generation!

---

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import glob

# Balancing techniques
from imblearn.over_sampling import SMOTE, ADASYN
from sklearn.model_selection import train_test_split

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ Libraries imported successfully")
print(f"  - NumPy: {np.__version__}")
print(f"  - Pandas: {pd.__version__}")
print("  - imbalanced-learn: Available")

---

## 📂 Step 1: Load Processed Features

We'll load the processed pose data from the previous steps.

In [ ]:
# Load all processed .npz files
data_dir = Path('../data/processed')
files = sorted(glob.glob(str(data_dir / '*.npz')))

print(f"Found {len(files)} processed files\n")

# Show sample files
print("Sample files:")
for f in files[:5]:
    print(f"  - {Path(f).name}")
print("  ...")

In [ ]:
# Parse data: Extract features and labels
X_sequences = []  # Variable-length sequences
y_labels = []     # Labels: exercise type and form
metadata = []     # Track original filenames

for file_path in files:
    filename = Path(file_path).stem
    
    # Parse filename: "exercise_form_id.npz"
    # Examples: "bicep_curls_correct_001.npz", "pushup_incorrect_005.npz"
    parts = filename.split('_')
    
    # Determine exercise type and form
    if 'correct' in filename:
        form_label = 'correct'
        exercise = filename.split('_correct')[0]
    elif 'incorrect' in filename:
        form_label = 'incorrect'
        exercise = filename.split('_incorrect')[0]
    else:
        # Legacy format: "curl_correct_001.npz"
        if 'curl' in filename:
            exercise = 'bicep_curls'
            form_label = 'correct' if 'correct' in filename else 'incorrect'
        else:
            continue
    
    # Normalize exercise names
    exercise = exercise.replace('curl', 'bicep_curls')
    
    # Load landmarks
    data = np.load(file_path)
    landmarks = data['landmarks']  # Shape: (n_frames, 132)
    
    X_sequences.append(landmarks)
    y_labels.append(f"{exercise}_{form_label}")
    metadata.append({
        'filename': filename,
        'exercise': exercise,
        'form': form_label,
        'n_frames': len(landmarks)
    })

print(f"\n✓ Loaded {len(X_sequences)} sequences")
print(f"  Shape example: {X_sequences[0].shape} (frames × landmarks)")

# Create DataFrame for analysis
df = pd.DataFrame(metadata)
df['label'] = y_labels

print(f"\nDataset Summary:")
print(df.head())

---

## 📊 Step 2: Analyze Class Imbalance

### Understanding the Problem

**What to Look For:**
1. **Exercise Type Distribution** - Do we have equal squats, push-ups, curls?
2. **Form Distribution** - More correct than incorrect examples?
3. **Imbalance Ratio** - How severe is the imbalance?

**Imbalance Severity Scale:**
```
Ratio 1:1   → Perfectly balanced ✓
Ratio 1:2   → Mild imbalance
Ratio 1:5   → Moderate imbalance (action needed)
Ratio 1:10  → Severe imbalance (critical!)
Ratio 1:100 → Extreme imbalance
```

In [ ]:
# Analyze class distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Exercise Type Distribution
exercise_counts = df['exercise'].value_counts()
axes[0].bar(exercise_counts.index, exercise_counts.values, color=['#FF6B6B', '#4ECDC4', '#FFE66D'])
axes[0].set_title('Exercise Type Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Exercise Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Add count labels
for i, (idx, val) in enumerate(exercise_counts.items()):
    axes[0].text(i, val + 1, str(val), ha='center', va='bottom', fontweight='bold')

# 2. Form Distribution (Correct vs Incorrect)
form_counts = df['form'].value_counts()
colors = ['#95E1D3', '#F38181']
axes[1].bar(form_counts.index, form_counts.values, color=colors)
axes[1].set_title('Form Distribution (Correct vs Incorrect)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Form Quality')
axes[1].set_ylabel('Count')

# Add count labels
for i, (idx, val) in enumerate(form_counts.items()):
    axes[1].text(i, val + 1, str(val), ha='center', va='bottom', fontweight='bold')

# 3. Combined Label Distribution
label_counts = df['label'].value_counts()
axes[2].barh(label_counts.index, label_counts.values, color=plt.cm.Set3(np.linspace(0, 1, len(label_counts))))
axes[2].set_title('Detailed Class Distribution', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Count')
axes[2].set_ylabel('Class')

# Add count labels
for i, (idx, val) in enumerate(label_counts.items()):
    axes[2].text(val + 0.5, i, str(val), va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Class Distribution Analysis:")
print("=" * 60)
print("\nExercise Types:")
print(exercise_counts)
print("\nForm Quality:")
print(form_counts)
print("\nDetailed Classes:")
print(label_counts)

In [ ]:
# Calculate imbalance ratios
print("\n⚠️  IMBALANCE ANALYSIS")
print("=" * 60)

# Overall form imbalance
correct_count = len(df[df['form'] == 'correct'])
incorrect_count = len(df[df['form'] == 'incorrect'])
imbalance_ratio = max(correct_count, incorrect_count) / min(correct_count, incorrect_count)

print(f"\n1. Form Quality Imbalance:")
print(f"   Correct:   {correct_count:3d} samples")
print(f"   Incorrect: {incorrect_count:3d} samples")
print(f"   Ratio:     1:{imbalance_ratio:.2f}")

if imbalance_ratio > 1.5:
    print(f"   ⚠️  IMBALANCED! (Ratio > 1.5)")
else:
    print(f"   ✓ Balanced")

# Per-class imbalance
print(f"\n2. Per-Class Imbalance:")
min_class_count = label_counts.min()
max_class_count = label_counts.max()
class_imbalance_ratio = max_class_count / min_class_count

print(f"   Majority class: {label_counts.idxmax()} ({max_class_count} samples)")
print(f"   Minority class: {label_counts.idxmin()} ({min_class_count} samples)")
print(f"   Ratio:          1:{class_imbalance_ratio:.2f}")

if class_imbalance_ratio > 2.0:
    print(f"   ⚠️  SEVERE IMBALANCE! (Ratio > 2.0)")
elif class_imbalance_ratio > 1.5:
    print(f"   ⚠️  MODERATE IMBALANCE (Ratio > 1.5)")
else:
    print(f"   ✓ Well balanced")

---

## 🚨 Why Is This a Problem?

### The Accuracy Paradox Explained

**Example with 90:10 Imbalance:**

```python
# Naive model that always predicts "correct"
def stupid_model(x):
    return "correct"

# Result:
Accuracy: 90% ← Looks great!
Recall for "incorrect": 0% ← Completely useless!
```

### Real-World Impact

**Workout Monitoring System:**
- **Goal:** Detect incorrect form to prevent injury
- **Imbalanced Model:** Misses all incorrect form examples
- **Result:** User gets injured → System failure!

**The Minority Class is Often Most Important:**
- Medical diagnosis: Disease (rare) vs Healthy (common)
- Fraud detection: Fraud (rare) vs Legitimate (common)
- Exercise monitoring: Incorrect (rare) vs Correct (common)

---

## 🔄 Solution: Synthetic Data Generation

Instead of collecting more data, we'll **generate synthetic examples** of the minority class.

---

---

## 🎨 Step 3: Prepare Data for Balancing

### Challenge: Variable-Length Sequences

**Problem:** Our videos have different lengths (30-200 frames)

**Solutions:**
1. **Padding**: Add zeros to reach max length
2. **Truncation**: Cut to fixed length
3. **Feature Aggregation**: Convert to fixed-size features

**We'll use Option 3** - Use the feature engineering module to extract fixed-size features.

In [ ]:
# Import feature engineering module
from src.preprocessing.feature_engineering import FeatureEngineer

# Initialize feature engineer
fe = FeatureEngineer()

print("✓ Feature engineering module loaded")
print("\nExtracting fixed-size features from variable-length sequences...")

# Extract engineered features
X_features = []
y_encoded = []

# Encode labels: 0=correct, 1=incorrect
label_map = {'correct': 0, 'incorrect': 1}

for i, (sequence, label) in enumerate(zip(X_sequences, y_labels)):
    # Extract features
    features = fe.engineer_features(sequence)
    X_features.append(features)
    
    # Encode form label (not exercise type)
    form = df.iloc[i]['form']
    y_encoded.append(label_map[form])
    
    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/{len(X_sequences)} sequences...", end='\r')

X = np.array(X_features)
y = np.array(y_encoded)

print(f"\n\n✓ Feature extraction complete!")
print(f"  X shape: {X.shape} (samples × features)")
print(f"  y shape: {y.shape}")
print(f"  Feature vector size: {X.shape[1]} features")

# Show class distribution
unique, counts = np.unique(y, return_counts=True)
print(f"\nClass Distribution:")
for label, count in zip(unique, counts):
    label_name = 'Correct' if label == 0 else 'Incorrect'
    print(f"  {label_name:10s} (class {label}): {count:3d} samples")

---

## 🔬 Step 4: SMOTE (Synthetic Minority Oversampling Technique)

### How SMOTE Works

**Algorithm:**
```
For each minority sample:
  1. Find k nearest neighbors (k=5 by default)
  2. Randomly select one neighbor
  3. Create synthetic sample along the line:
     synthetic = sample + λ × (neighbor - sample)
     where λ ∈ [0, 1] is random
```

**Visual Intuition:**
```
Original:              After SMOTE:
  X                      X
                        / \
      X                X   X
                      /     \
Minority samples     X       X
```

### Why SMOTE?

**Advantages:**
1. **Not Just Duplication** - Creates new, realistic examples
2. **Interpolation** - New samples lie in feature space between existing samples
3. **Generalizes Better** - Model learns from diverse synthetic examples
4. **Proven Effective** - Industry standard for imbalanced learning

**vs. Random Oversampling:**
```
Random Oversampling:  [A, B, C] → [A, B, C, A, B, C] (exact copies)
SMOTE:               [A, B, C] → [A, B, C, A', B', C'] (new examples)
```

**Why Interpolation Works:**
- Similar exercises have similar pose patterns
- Feature space is continuous
- Interpolated samples are realistic

---

In [ ]:
# Apply SMOTE
print("Applying SMOTE...\n")

# Initialize SMOTE
smote = SMOTE(
    sampling_strategy='auto',  # Balance to majority class
    k_neighbors=5,              # Use 5 nearest neighbors
    random_state=42
)

# Apply SMOTE
X_smote, y_smote = smote.fit_resample(X, y)

print("✓ SMOTE complete!\n")
print("=" * 60)
print("BEFORE SMOTE:")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    label_name = 'Correct' if label == 0 else 'Incorrect'
    print(f"  {label_name:10s}: {count:3d} samples")

print("\nAFTER SMOTE:")
unique, counts = np.unique(y_smote, return_counts=True)
for label, count in zip(unique, counts):
    label_name = 'Correct' if label == 0 else 'Incorrect'
    print(f"  {label_name:10s}: {count:3d} samples")

print("\n" + "=" * 60)
print(f"\nNew samples generated: {len(X_smote) - len(X)}")
print(f"Total dataset size: {len(X)} → {len(X_smote)}")

In [ ]:
# Visualize SMOTE effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before SMOTE
unique, counts = np.unique(y, return_counts=True)
labels = ['Correct', 'Incorrect']
colors = ['#95E1D3', '#F38181']

axes[0].bar(labels, counts, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[0].set_title('BEFORE SMOTE', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Samples', fontsize=12)
axes[0].set_ylim(0, max(counts) * 1.2)

# Add count labels
for i, count in enumerate(counts):
    axes[0].text(i, count + 1, str(count), ha='center', va='bottom', fontweight='bold', fontsize=12)

# Add imbalance ratio
ratio = max(counts) / min(counts)
axes[0].text(0.5, max(counts) * 1.1, f'Imbalance Ratio: 1:{ratio:.2f}',
             ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

# After SMOTE
unique, counts = np.unique(y_smote, return_counts=True)

axes[1].bar(labels, counts, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[1].set_title('AFTER SMOTE', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Number of Samples', fontsize=12)
axes[1].set_ylim(0, max(counts) * 1.2)

# Add count labels
for i, count in enumerate(counts):
    axes[1].text(i, count + 1, str(count), ha='center', va='bottom', fontweight='bold', fontsize=12)

# Add balanced indicator
axes[1].text(0.5, max(counts) * 1.1, '✓ BALANCED (1:1)',
             ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

plt.suptitle('SMOTE Balancing Effect', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 🎯 Step 5: ADASYN (Adaptive Synthetic Sampling)

### How ADASYN Differs from SMOTE

**Key Innovation:** Focus on **hard-to-learn** examples

**Algorithm:**
```
1. Calculate density distribution:
   For each minority sample, count majority neighbors
   
2. Normalize to get sampling weights:
   Samples in "difficult" regions get higher weights
   
3. Generate synthetic samples:
   More samples in sparse/difficult regions
   Fewer samples in dense/easy regions
```

### Why ADASYN?

**SMOTE Limitation:**
```
Easy region:      O O O O O    ← SMOTE adds same number here
                  X X X X

Hard region:      O O O O O    ← and here (but this needs more!)
                  X
```

**ADASYN Solution:**
```
Easy region:      O O O O O    ← Add fewer samples
                  X X X X X

Hard region:      O O O O O    ← Add MORE samples (adaptive!)
                  X X X X X X
```

### When to Use ADASYN?

**ADASYN is better when:**
- Minority class has **borderline cases** (hard to classify)
- Class overlap is significant
- You want to focus model attention on difficult examples

**SMOTE is better when:**
- Classes are well-separated
- Uniform oversampling is desired
- Simpler is better (Occam's Razor)

### Real-World Analogy

**Learning Math:**
- **SMOTE**: Practice all problems equally
- **ADASYN**: Practice harder problems MORE

**Result:** ADASYN often produces better decision boundaries!

---

In [ ]:
# Apply ADASYN
print("Applying ADASYN...\n")

# Initialize ADASYN
adasyn = ADASYN(
    sampling_strategy='auto',
    n_neighbors=5,
    random_state=42
)

# Apply ADASYN
try:
    X_adasyn, y_adasyn = adasyn.fit_resample(X, y)
    
    print("✓ ADASYN complete!\n")
    print("=" * 60)
    print("BEFORE ADASYN:")
    unique, counts = np.unique(y, return_counts=True)
    for label, count in zip(unique, counts):
        label_name = 'Correct' if label == 0 else 'Incorrect'
        print(f"  {label_name:10s}: {count:3d} samples")
    
    print("\nAFTER ADASYN:")
    unique, counts = np.unique(y_adasyn, return_counts=True)
    for label, count in zip(unique, counts):
        label_name = 'Correct' if label == 0 else 'Incorrect'
        print(f"  {label_name:10s}: {count:3d} samples")
    
    print("\n" + "=" * 60)
    print(f"\nNew samples generated: {len(X_adasyn) - len(X)}")
    print(f"Total dataset size: {len(X)} → {len(X_adasyn)}")
    
except Exception as e:
    print(f"⚠️  ADASYN failed: {e}")
    print("\nNote: ADASYN may fail if:")
    print("  - Not enough minority samples")
    print("  - Feature space is too sparse")
    print("\nFalling back to SMOTE...")
    X_adasyn, y_adasyn = X_smote, y_smote

In [ ]:
# Compare SMOTE vs ADASYN
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

labels = ['Correct', 'Incorrect']
colors = ['#95E1D3', '#F38181']

# Original
unique, counts = np.unique(y, return_counts=True)
axes[0].bar(labels, counts, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[0].set_title('Original Data', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Samples', fontsize=12)
for i, count in enumerate(counts):
    axes[0].text(i, count + 1, str(count), ha='center', va='bottom', fontweight='bold')

# SMOTE
unique, counts = np.unique(y_smote, return_counts=True)
axes[1].bar(labels, counts, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[1].set_title('SMOTE Balanced', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Number of Samples', fontsize=12)
for i, count in enumerate(counts):
    axes[1].text(i, count + 1, str(count), ha='center', va='bottom', fontweight='bold')

# ADASYN
unique, counts = np.unique(y_adasyn, return_counts=True)
axes[2].bar(labels, counts, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[2].set_title('ADASYN Balanced', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Number of Samples', fontsize=12)
for i, count in enumerate(counts):
    axes[2].text(i, count + 1, str(count), ha='center', va='bottom', fontweight='bold')

plt.suptitle('Comparison: Original vs SMOTE vs ADASYN', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print comparison
print("\n📊 COMPARISON SUMMARY")
print("=" * 60)
print(f"{'Method':<15} {'Total Samples':<15} {'Minority Count':<15} {'Ratio'}")
print("-" * 60)
print(f"{'Original':<15} {len(X):<15} {min(np.bincount(y)):<15} 1:{max(np.bincount(y))/min(np.bincount(y)):.2f}")
print(f"{'SMOTE':<15} {len(X_smote):<15} {min(np.bincount(y_smote)):<15} 1:{max(np.bincount(y_smote))/min(np.bincount(y_smote)):.2f}")
print(f"{'ADASYN':<15} {len(X_adasyn):<15} {min(np.bincount(y_adasyn)):<15} 1:{max(np.bincount(y_adasyn))/min(np.bincount(y_adasyn)):.2f}")

---

## 📚 Step 6: Train/Test Split Strategy

### Critical Rule: Balance ONLY Training Data!

**Why?**

```
❌ WRONG: Balance entire dataset, then split
  Problem: Synthetic samples in test set → Inflated performance!

✓ CORRECT: Split first, then balance training set only
  Reason: Test set must reflect REAL WORLD distribution
```

### The Evaluation Philosophy

**Training Set (Balanced):**
- Model learns from balanced examples
- Pays equal attention to both classes
- Synthetic samples are OK here

**Test Set (Original Distribution):**
- Reflects real-world data
- No synthetic samples
- Honest performance estimate

### Stratified Split

**What:** Maintain class proportions in train/test splits

**Why:** Ensures test set represents all classes fairly

**Example:**
```
Original: 80% correct, 20% incorrect

Stratified Split:
  Train: 80% correct, 20% incorrect
  Test:  80% correct, 20% incorrect

Random Split (might be):
  Train: 85% correct, 15% incorrect
  Test:  70% correct, 30% incorrect ← Not representative!
```

---

In [ ]:
# CORRECT APPROACH: Split FIRST, then balance training set only
print("Splitting data with stratification...\n")

# Split into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,  # Maintain class proportions
    random_state=42
)

print("✓ Initial split complete")
print(f"  Training set: {len(X_train)} samples")
print(f"  Test set:     {len(X_test)} samples")

# Now balance ONLY training set with SMOTE
print("\nBalancing training set with SMOTE...")
smote_train = SMOTE(sampling_strategy='auto', k_neighbors=5, random_state=42)
X_train_balanced, y_train_balanced = smote_train.fit_resample(X_train, y_train)

print("\n" + "=" * 60)
print("TRAINING SET:")
print("  Before SMOTE:")
unique, counts = np.unique(y_train, return_counts=True)
for label, count in zip(unique, counts):
    label_name = 'Correct' if label == 0 else 'Incorrect'
    print(f"    {label_name:10s}: {count:3d} samples")

print("\n  After SMOTE:")
unique, counts = np.unique(y_train_balanced, return_counts=True)
for label, count in zip(unique, counts):
    label_name = 'Correct' if label == 0 else 'Incorrect'
    print(f"    {label_name:10s}: {count:3d} samples")

print("\nTEST SET (Original Distribution):")
unique, counts = np.unique(y_test, return_counts=True)
for label, count in zip(unique, counts):
    label_name = 'Correct' if label == 0 else 'Incorrect'
    print(f"  {label_name:10s}: {count:3d} samples")

print("\n" + "=" * 60)
print("\n✓ Train/Test split complete!")
print(f"  Training set (balanced): {len(X_train_balanced)} samples")
print(f"  Test set (original):     {len(X_test)} samples")

In [ ]:
# Visualize train/test split strategy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = ['Correct', 'Incorrect']
colors = ['#95E1D3', '#F38181']

# Training Set (Balanced)
unique, counts = np.unique(y_train_balanced, return_counts=True)
axes[0].bar(labels, counts, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[0].set_title('Training Set (SMOTE Balanced)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Samples', fontsize=12)
for i, count in enumerate(counts):
    axes[0].text(i, count + 1, str(count), ha='center', va='bottom', fontweight='bold')
axes[0].text(0.5, max(counts) * 0.9, '✓ Balanced for Training',
             ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

# Test Set (Original)
unique, counts = np.unique(y_test, return_counts=True)
axes[1].bar(labels, counts, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[1].set_title('Test Set (Original Distribution)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Number of Samples', fontsize=12)
for i, count in enumerate(counts):
    axes[1].text(i, count + 1, str(count), ha='center', va='bottom', fontweight='bold')
axes[1].text(0.5, max(counts) * 0.9, 'Real-world Distribution',
             ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

plt.suptitle('Train/Test Split Strategy', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 💾 Step 7: Save Balanced Dataset

Save the balanced training data and original test data for model training.

In [ ]:
# Save balanced datasets
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

# Save training set (balanced)
np.savez_compressed(
    output_dir / 'train_balanced_smote.npz',
    X=X_train_balanced,
    y=y_train_balanced
)

# Save test set (original distribution)
np.savez_compressed(
    output_dir / 'test_original.npz',
    X=X_test,
    y=y_test
)

# Save original training set (for comparison)
np.savez_compressed(
    output_dir / 'train_original.npz',
    X=X_train,
    y=y_train
)

print("✓ Datasets saved!\n")
print("Files created:")
print(f"  - {output_dir / 'train_balanced_smote.npz'}")
print(f"  - {output_dir / 'train_original.npz'}")
print(f"  - {output_dir / 'test_original.npz'}")

# Print file sizes
print("\nFile sizes:")
for filename in ['train_balanced_smote.npz', 'train_original.npz', 'test_original.npz']:
    filepath = output_dir / filename
    size_kb = filepath.stat().st_size / 1024
    print(f"  {filename:30s}: {size_kb:8.1f} KB")

---

## 📊 Step 8: Summary and Recommendations

### Which Balancing Method to Use?

#### SMOTE (Recommended for Most Cases)

**Use SMOTE when:**
- ✓ General-purpose balancing needed
- ✓ Classes are reasonably separated
- ✓ You want consistent, predictable behavior
- ✓ Dataset is small to medium size

**Advantages:**
- Stable and well-tested
- Works across diverse datasets
- Easy to understand and explain
- Good default choice

#### ADASYN (Use for Difficult Cases)

**Use ADASYN when:**
- ✓ Significant class overlap exists
- ✓ Some minority samples are harder to learn
- ✓ You want adaptive sampling density
- ✓ Willing to tune for better performance

**Advantages:**
- Focuses on difficult decision boundaries
- Better for borderline cases
- Can improve performance on hard examples

**Disadvantages:**
- May fail with very small datasets
- Less predictable than SMOTE
- Slightly more complex

### Our Recommendation: SMOTE

**For this project:**
- Dataset is relatively small
- Classes (correct/incorrect form) should be separable
- SMOTE provides good balance between complexity and effectiveness

### Alternative Approaches

1. **Class Weights:**
   ```python
   # Penalize misclassifying minority class more
   model.fit(X, y, class_weight='balanced')
   ```
   - Pros: No synthetic data, maintains dataset size
   - Cons: Model-dependent, may not work as well

2. **Undersampling:**
   ```python
   # Remove majority class samples
   from imblearn.under_sampling import RandomUnderSampler
   ```
   - Pros: No synthetic data
   - Cons: Loses valuable data, reduces dataset size

3. **Ensemble Methods:**
   ```python
   # BalancedRandomForest, EasyEnsemble
   from imblearn.ensemble import BalancedRandomForestClassifier
   ```
   - Pros: Built-in balancing
   - Cons: Limited to specific models

### Key Takeaways

1. **Always balance training data only** - Never test set!
2. **Use stratified splits** - Maintain class proportions
3. **SMOTE is a good default** - Works for most cases
4. **Evaluate with appropriate metrics** - Not just accuracy!
5. **Consider class weights** - As alternative/complement to SMOTE

---

## 🎓 Next Steps

Now that we have balanced data, we can proceed to:

1. **Model Training** (Notebook 5)
   - Train LSTM, Random Forest, SVM
   - Compare performance on balanced data
   - Use appropriate evaluation metrics (not just accuracy!)

2. **Evaluation Metrics to Use:**
   - Precision: Of predicted incorrect, how many are truly incorrect?
   - Recall: Of actual incorrect, how many did we detect?
   - F1-Score: Harmonic mean of precision and recall
   - Confusion Matrix: Detailed breakdown of predictions

---

**End of Notebook 4: Data Balancing**